# 예외처리가 없는 VOCAB

* **토크나이저**는 사람이 쓰는 텍스트를 인공지능 모델이 이해할 수 있는 '토큰'이라는 작은 단위로 쪼개고, 이를 숫자로 바꾸는 자연어 처리(NLP) 핵심 도구

* **임베딩**은 토크나이저가 쪼갠 토큰(숫자 ID)들을 컴퓨터가 의미를 이해할 수 있도록 고차원의 연속적인 벡터(숫자 배열)로 변환하는 기술

In [9]:
text = "나는 밥을 먹었다. 너는 밥을 먹었니?"

# vocab 생성
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

# encode(문자열 -> 정수 리스트)
# 보통 모델이 학습할 수 있는 데이터로 만들 때 사용
def encode(s):
    return [stoi[ch] for ch in s]

# decode(정수 리스트 -> 문자열)
# 보통 모델이 예측한 값을 인간의 언어로 바꿀 때 사용
def decode(ids):
    return "".join(itos[i] for i in ids)

print(encode("나는"))
print(decode(encode("나는")))

[3, 5]
나는


In [10]:
# 공백 기준 분리가 아님
print(chars)

[' ', '.', '?', '나', '너', '는', '니', '다', '먹', '밥', '었', '을']


In [11]:
print(stoi)

{' ': 0, '.': 1, '?': 2, '나': 3, '너': 4, '는': 5, '니': 6, '다': 7, '먹': 8, '밥': 9, '었': 10, '을': 11}


In [12]:
print(itos)

{0: ' ', 1: '.', 2: '?', 3: '나', 4: '너', 5: '는', 6: '니', 7: '다', 8: '먹', 9: '밥', 10: '었', 11: '을'}


In [ ]:
# 아래는 예외처리가 없는 경우 발생하는 오류
# 이는 전형적인 한계
encode("나도 그렇게 생각해.")

KeyError: '도'

# 예외처리가 있는 VOCAB

In [ ]:
sentences = ["나는 밥을 먹었다", "너는 밥을 먹었니"]

# 특수 토큰: 패딩, 문장 시작/끝, 미등록 단어
SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]

words = sorted(set(w for s in sentences for w in s.split()))
vocab = SPECIALS + words # 특수 토큰들에 대해서도 index를 매기기 위해서 SPECIALS와 words를 합침.
stoi = {w: i for i, w in enumerate(vocab)}

def encode(sentence):
    # 예외처리가 동작하면서 이전에 없던 값들이면 <unk>로 대체.
    ids = [stoi.get(w, stoi["<unk>"]) for w in sentence.split()]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]

print(encode("나는 밥을 먹었다"))

[1, 4, 8, 7, 2]


In [16]:
print(words)

['나는', '너는', '먹었니', '먹었다', '밥을']


In [17]:
print(vocab)

['<pad>', '<sos>', '<eos>', '<unk>', '나는', '너는', '먹었니', '먹었다', '밥을']


In [18]:
print(stoi)

{' ': 0, '.': 1, '?': 2, '나': 3, '너': 4, '는': 5, '니': 6, '다': 7, '먹': 8, '밥': 9, '었': 10, '을': 11}


# EMBEDDING

In [ ]:
import torch.nn as nn

vocab_size = 50
embed_dim = 16

# embedding layer
embedding = nn.Embedding(vocab_size, embed_dim)

In [20]:
text = ["안녕", "나는 오늘 밥을 먹었어."]

# 그대로 넣으면 좋겠지만, 당연하게도 에러가 남.
# embedding도 하나의 layer이기 때문에.
embedding(text)

TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not list

In [50]:
import torch

# 따라서 text -> 모델로 가는 과정에서 사이에 전처리를 해줘야함.
# 이는 text -> preprocessing -> model로 표현할 수 있음.
# 앞에서 배운 encode 함수를 활용할 수 있음.

text = ["안녕", "나는 오늘 밥을 먹었어."]

# vocab 사전을 만들기 위한 데이터
sentences = ["나는 밥을 먹었다", "너는 밥을 먹었니"]

# 특수 토큰: 패딩, 문장 시작/끝, 미등록 단어
SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]

# sentences에서 공백을 기준으로 하나의 단어로 판별
# 여기서 vocab 사전을 만듬
words = sorted(set(w for s in sentences for w in s.split()))
vocab = SPECIALS + words # 특수 토큰들에 대해서도 index를 매기기 위해서 SPECIALS와 words를 합침.
stoi = {w: i for i, w in enumerate(vocab)}

# 앞에서 encode 함수는 string에 대해서 처리
def encode(sentence):
    # 예외처리가 동작하면서 이전에 없던 값들이면 <unk>로 대체.
    ids = [stoi.get(w, stoi["<unk>"]) for w in sentence.split()]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]

# 실제로는 리스트 형태로 들어옴(텐서)
# 이에 맞춰서 preprocessing 함수를 새로 선언
def preprocessing(text, padding_size=10, padding_token=0):
    input_tokens = []
    for sentence in text:
        # 우선 각 토큰들을 encoding
        print("##### Raw 데이터 #####")
        print(sentence)

        encoded_sentence = encode(sentence)
        print("##### Encoding된 데이터 #####")
        print(encoded_sentence)

        # embedding에 들어갈 때는 모든 데이터의 크기가 동일해야 하므로, padding적용
        encoded_padded_sentence = encoded_sentence + [padding_token] * (padding_size - len(encoded_sentence))
        print("##### Padding된 데이터 #####")
        print(encoded_padded_sentence)

        input_tokens.append(encoded_padded_sentence)
        print()

    print("##### 최종 embedding에 들어갈 데이터 #####")
    print(torch.tensor(input_tokens))
    print()

    return torch.tensor(input_tokens) # tensor 적용은 반드시 해야함.

preprocessing(text)

##### Raw 데이터 #####
안녕
##### Encoding된 데이터 #####
[1, 3, 2]
##### Padding된 데이터 #####
[1, 3, 2, 0, 0, 0, 0, 0, 0, 0]

##### Raw 데이터 #####
나는 오늘 밥을 먹었어.
##### Encoding된 데이터 #####
[1, 4, 3, 8, 3, 2]
##### Padding된 데이터 #####
[1, 4, 3, 8, 3, 2, 0, 0, 0, 0]

##### 최종 embedding에 들어갈 데이터 #####
tensor([[1, 3, 2, 0, 0, 0, 0, 0, 0, 0],
        [1, 4, 3, 8, 3, 2, 0, 0, 0, 0]])



tensor([[1, 3, 2, 0, 0, 0, 0, 0, 0, 0],
        [1, 4, 3, 8, 3, 2, 0, 0, 0, 0]])

In [54]:
display(stoi)

{'<pad>': 0,
 '<sos>': 1,
 '<eos>': 2,
 '<unk>': 3,
 '나는': 4,
 '너는': 5,
 '먹었니': 6,
 '먹었다': 7,
 '밥을': 8}

In [ ]:
# raw data -> preprocessing -> embedding -> model

import torch.nn as nn

text = ["안녕", "나는 오늘 밥을 먹었어."]

vocab_size = 50
embed_dim = 16

# embedding layer
embedding = nn.Embedding(vocab_size, embed_dim)

# 전처리
input_data = preprocessing(text)
print(input_data)

# embedding을 거쳐서 모델은 문장을 이해
embedded_input_data = embedding(input_data)

##### Raw 데이터 #####
안녕
##### Encoding된 데이터 #####
[1, 3, 2]
##### Padding된 데이터 #####
[1, 3, 2, 0, 0, 0, 0, 0, 0, 0]

##### Raw 데이터 #####
나는 오늘 밥을 먹었어.
##### Encoding된 데이터 #####
[1, 4, 3, 8, 3, 2]
##### Padding된 데이터 #####
[1, 4, 3, 8, 3, 2, 0, 0, 0, 0]

##### 최종 embedding에 들어갈 데이터 #####
tensor([[1, 3, 2, 0, 0, 0, 0, 0, 0, 0],
        [1, 4, 3, 8, 3, 2, 0, 0, 0, 0]])

tensor([[1, 3, 2, 0, 0, 0, 0, 0, 0, 0],
        [1, 4, 3, 8, 3, 2, 0, 0, 0, 0]])


In [ ]:
# "안녕"이라는 문장에 대해 embedding은 아래와 같이 이해한다.
embedded_input_data[0]

tensor([[-1.0036e+00, -9.4424e-01, -5.4276e-04,  1.3206e+00,  5.9247e-01,
         -8.9781e-01, -2.7243e-01, -1.2058e+00,  1.1261e+00, -1.2647e+00,
         -1.0608e+00,  1.0430e+00, -4.3016e-01,  1.8314e-01,  1.9459e+00,
          1.2876e+00],
        [-2.7004e-01,  1.1348e+00,  4.0311e-01,  3.0169e-01,  2.8118e-01,
         -8.7715e-01,  1.7979e+00,  9.1154e-01, -1.7082e+00, -8.4345e-01,
         -6.4321e-01, -1.8813e-01,  6.4100e-01,  4.9392e-01,  3.1630e-01,
         -1.6709e+00],
        [-4.7456e-01,  1.8839e-01, -8.3402e-01, -1.7395e+00,  9.4685e-01,
         -3.4997e-01, -1.0962e+00,  1.1774e+00,  1.5018e+00,  2.1898e-01,
          4.7552e-01, -1.2411e+00,  2.0499e+00,  1.3596e+00,  1.7072e+00,
         -1.3001e-02],
        [-1.1030e+00,  5.3450e-01,  1.3117e+00, -8.1452e-01, -5.8488e-01,
         -2.0892e-01, -8.4249e-01, -1.0237e+00, -2.0315e-01, -1.7176e+00,
         -4.4550e-01,  7.4953e-01, -1.9142e-01, -2.3374e-01,  3.1709e-01,
          2.6069e-01],
        [-1.1030e+00

In [62]:
embedded_input_data[0].shape

torch.Size([10, 16])

In [58]:
embedded_input_data.shape

torch.Size([2, 10, 16])

In [ ]:
'''
nn.Embedding에 들어가는 vocab_size는 vocab_size x dim 형태의 행렬을 반환하는 것이 아님.
vocab_size까지의 인덱스가 들어오는 것임.
'''

In [66]:
# 즉, 아래는 에러가 남.
input_data = torch.tensor([50])
embedding(input_data)

IndexError: index out of range in self

### 아래 링크에서 Embedding을 보다 직관적으로 이해할 수 있습니다 !

woman과 근처에 있는 단어를 찾아봅시다.

https://projector.tensorflow.org/
